In [1]:
import numpy as np
import time
import sys
sys.path.insert(0, '../..')

from sn_clf.scripts.utils import load_features, get_sn_label_from_akb, plot_config

In [2]:
feature_names = '../../dr23-features/feature_snad_clf_r_100.name'
with open(feature_names) as f:
    names = f.read().split()

oids, init_features = load_features('../../dr23-features/sid_snad_clf_r_100.dat', '../../dr23-features/feature_snad_clf_r_100.dat')
filtered = np.array([False if 'chi2' in str(item) else True for item in names])
proba_file = '../../dr23-features/SNclf_proba_no-chi2.dat'
sn_score = np.memmap(proba_file, mode='c', dtype=np.float32)

In [6]:
no_chi_names = [item for item in names if 'chi2' not in str(item)] + ['SN_clf_proba']
with open('../../dr23-features/feature_sn_no-chi2.name', "w") as file:
    for item in names:
        file.write(f"{item}\n")

In [5]:
t = time.monotonic()
features = np.hstack([init_features[:, filtered], sn_score.reshape(-1,1)])
t = (time.monotonic() - t) / 60
print(f'Concat in {t:.0f}')

Concat in 13


In [10]:
# 1. Определяем имя файла и параметры
output_file = "../../dr23-features/feature_sn_no-chi2.dat"
dtype = np.float32  # тот же dtype, что и у proba

# 2. Сохраняем массив в бинарном формате (memmap-совместимом)
t_start = time.monotonic()

# Создаем memmap-файл в режиме записи ('w+')
memmap_shape = features.shape  # (N, M)
memmap_array = np.memmap(
    output_file,
    dtype=dtype,
    mode='w+',  # режим записи (создаст файл)
    shape=memmap_shape,
)

# Копируем данные из RAM в файл (построчно, если массив огромный)
memmap_array[:] = features[:]

# Принудительно записываем изменения на диск
memmap_array.flush()
del memmap_array  # закрываем файл

t_total = (time.monotonic() - t_start) / 60
print(f"Массив сохранен в {output_file} за {t_total:.1f} мин.")

Массив сохранен в ../../dr23-features/features_sn_no-chi2.dat за 5.1 мин.
